Task1 FIAT

The main problem is trying to calculate how many cars each models should be produced so that it can get the maximun profit. Therefore, we need to figure out the net profit that each model will get.

In [3]:
from stat import FILE_ATTRIBUTE_ARCHIVE

models=["Panda", "500", "Musa", "Giulia"]
price={"Panda":106, "500":136, "Musa":150, "Giulia":427}
tax={"Panda":0.30, "500":0.15, "Musa":0.20, "Giulia":0.30}
material= {"Panda": 0.57, "500": 0.60, "Musa": 0.55, "Giulia": 0.45}
salary={"Panda": 20000, "500": 11000, "Musa": 20000, "Giulia": 26000}
manhour={"Panda": 40, "500": 45, "Musa": 38, "Giulia": 100}
min_prod={"Panda": 120000, "500": 100000, "Musa": 80000, "Giulia": 15000}
hour_per_month=160
c={} #net income
cost={} # cost
for m in models:
    material_cost=price[m]*material[m]
    labor_cost_per_hour=salary[m]/hour_per_month/1000
    labor_cost=labor_cost_per_hour*manhour[m]
    pretax_profit=price[m]-material_cost-labor_cost
    net_profit=pretax_profit*(1-tax[m])
    c[m] = net_profit
    cost[m]=labor_cost+material_cost
    print(f"{m}: material={material_cost:.2f}, labor={labor_cost:.2f}, "
          f"pretax={pretax_profit:.2f}, net_profit={net_profit:.2f}")

Panda: material=60.42, labor=5.00, pretax=40.58, net_profit=28.41
500: material=81.60, labor=3.09, pretax=51.31, net_profit=43.61
Musa: material=82.50, labor=4.75, pretax=62.75, net_profit=50.20
Giulia: material=192.15, labor=16.25, pretax=218.60, net_profit=153.02


The objective function is

max Z=c1x1+c2x2+c3x3+c4x4

with some cost constraint


In [4]:
import gurobipy as gp
from gurobipy import GRB
m=gp.Model("FIAT")
x=m.addVars(models,name='x')
m.setObjective(gp.quicksum(c[i] * x[i] for i in models), GRB.MAXIMIZE)
m.addConstr(gp.quicksum(cost[i] * x[i] for i in models) <= 4e7, "budget")
for i in models:
    m.addConstr(x[i] >= min_prod[i], f"minprod_{i}")
m.addConstr(x["Panda"] + x["Musa"] <= 300000, "marketing")
m.optimize()
for i in models:
    print(i, x[i].X)

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 6 rows, 4 columns and 10 nonzeros (Max)
Model fingerprint: 0xa94ff6bc
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [3e+01, 2e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 4e+07]

Presolve removed 6 rows and 4 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.4048076e+07   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Optimal objective  2.404807577e+07
Panda 120000.0
500 100000.0
Musa 80000.0
Giulia 80135.43666026872


In [6]:
# 查看约束的右端项(RHS)允许变化的范围，不影响最优基/解结构
for c in m.getConstrs():
    print(c.ConstrName, "RHS =", c.RHS,
          "range:[", c.SARHSLow, ",", c.SARHSUp, "]")

# 查看目标函数系数(即c_i)允许变化的范围
for v in m.getVars():
    print(v.VarName, "coef =", v.Obj,
          "range:[", v.SAObjLow, ",", v.SAObjUp, "]")

budget RHS = 40000000.0 range:[ 26425775.0 , inf ]
minprod_Panda RHS = 120000.0 range:[ -0.0 , 220000.0 ]
minprod_500 RHS = 100000.0 range:[ -0.0 , 260274.2233045532 ]
minprod_Musa RHS = 80000.0 range:[ -0.0 , 180000.0 ]
minprod_Giulia RHS = 15000.0 range:[ -inf , 80135.43666026872 ]
marketing RHS = 300000.0 range:[ 200000.0 , inf ]
x[Panda] coef = 28.406000000000002 range:[ -inf , 48.035357005758144 ]
x[500] coef = 43.610312500000006 range:[ -inf , 62.18732065738963 ]
x[Musa] coef = 50.2 range:[ -inf , 64.0642754318618 ]
x[Giulia] coef = 153.01999999999998 range:[ 119.9046418338109 , inf ]


Task2 The Global Market

We need to recgonize the decision variables, x_ij, i represents the model, j represents the country that the cars will be sold. Therefore, i belongs to the set { Panda, 500, Musa, Guilia}, j belongs to the set {Sweden, Italy, Poland, US}. X_ij is the number that each model should be sold in different country.
Thus, we need to calculate the net profit that per unit model sold in the 4 countries.

In [47]:
# Task II: 用 home price 重新计算材料成本

home_price = {"Panda": 106, "500": 92, "Musa": 150, "Giulia": 427}  # 500改成波兰价格
models=["Panda", "500", "Musa", "Giulia"]
tax={"Panda":0.30, "500":0.15, "Musa":0.20, "Giulia":0.30}
material= {"Panda": 0.57, "500": 0.60, "Musa": 0.55, "Giulia": 0.45}
salary={"Panda": 20000, "500": 11000, "Musa": 20000, "Giulia": 26000}
manhour={"Panda": 40, "500": 45, "Musa": 38, "Giulia": 100}

hour_per_month=160
cost_taskII = {}
for i in models:
    material_cost = home_price[i] * material[i]
    labor_cost = (salary[i] / hour_per_month / 1000) * manhour[i]
    cost_taskII[i] = material_cost + labor_cost
    print(f"{i}: material={material_cost:.3f}, labor={labor_cost:.3f}, total_cost={cost_taskII[i]:.3f}")


Panda: material=60.420, labor=5.000, total_cost=65.420
500: material=55.200, labor=3.094, total_cost=58.294
Musa: material=82.500, labor=4.750, total_cost=87.250
Giulia: material=192.150, labor=16.250, total_cost=208.400


In [55]:

models = ["Panda", "500", "Musa", "Giulia"]
countries = ["Poland", "Italy", "US", "Sweden"]
data = {
    ("Panda","Poland"):  (0.8,  86,  75),
    ("Panda","Italy"):   (0,    106, 35),
    ("Panda","US"):      (3.5, 150, 40),
    ("Panda","Sweden"):  (2, 112, 2),

    ("500","Poland"):    (0,    92,  20),
    ("500","Italy"):     (1, 136, 40),
    ("500","US"):        (2.8, 170, 50),
    ("500","Sweden"):    (1.6, 150, 5),

    ("Musa","Poland"):   (12,   100, 10),
    ("Musa","Italy"):    (0,    150, 80),

    ("Musa","Sweden"):   (2.2, 170, 1),



    ("Giulia","Italy"):  (0,    427, 8),
    ("Giulia","US"):     (5, 550, 3),
    ("Giulia","Sweden"): (2.5, 500, 1),
}
c_market = {}
demand = {}
total_cost_market = {}
for (i,j),(dl,mkt_price,dm) in data.items():
    if j=="US":
       export_tax=0.025*mkt_price
    else:
        export_tax=0
    total_cost_market[(i, j)] = cost_taskII[i] + dl
    pretax_profit=mkt_price-total_cost_market[(i, j)]-export_tax
    net_profit=pretax_profit*(1-tax[i])
    c_market[(i, j)] = net_profit
    demand[(i, j)] = dm
    print(f"{i:8s} -> {j:8s}: cost(material+labor)={cost[i]:.3f} kkr, "
          f"delivery={dl:.3f} kkr, export_tax={export_tax:.3f} kkr, "
          f"pretax={pretax_profit:.3f} kkr, net_profit={net_profit:.3f} kkr, demand={dm}")


Panda    -> Poland  : cost(material+labor)=65.420 kkr, delivery=0.800 kkr, export_tax=0.000 kkr, pretax=19.780 kkr, net_profit=13.846 kkr, demand=75
Panda    -> Italy   : cost(material+labor)=65.420 kkr, delivery=0.000 kkr, export_tax=0.000 kkr, pretax=40.580 kkr, net_profit=28.406 kkr, demand=35
Panda    -> US      : cost(material+labor)=65.420 kkr, delivery=3.500 kkr, export_tax=3.750 kkr, pretax=77.330 kkr, net_profit=54.131 kkr, demand=40
Panda    -> Sweden  : cost(material+labor)=65.420 kkr, delivery=2.000 kkr, export_tax=0.000 kkr, pretax=44.580 kkr, net_profit=31.206 kkr, demand=2
500      -> Poland  : cost(material+labor)=84.694 kkr, delivery=0.000 kkr, export_tax=0.000 kkr, pretax=33.706 kkr, net_profit=28.650 kkr, demand=20
500      -> Italy   : cost(material+labor)=84.694 kkr, delivery=1.000 kkr, export_tax=0.000 kkr, pretax=76.706 kkr, net_profit=65.200 kkr, demand=40
500      -> US      : cost(material+labor)=84.694 kkr, delivery=2.800 kkr, export_tax=4.250 kkr, pretax=104

The objectibe function is

Z= sum c_ij*x_ij

In [56]:
import gurobipy as gp
from gurobipy import GRB
m2=gp.Model("Grobal market")
# task 1 requirement
models = ["Panda", "500", "Musa", "Giulia"]
pairs = list(data.keys())
# variables：x[i,j] = the number of the model sold to each country
x = m2.addVars(pairs, name="ship")
# objective function：maximize the net profit
m2.setObjective(gp.quicksum(c_market[p] * x[p] for p in pairs), GRB.MAXIMIZE)
# constraints
for p in pairs:
    m2.addConstr(x[p] >= demand[p], f"demand_{p[0]}_{p[1]}")

m2.addConstr(
    gp.quicksum(total_cost_market[p] * x[p] for p in pairs) <= 4e4,
    name="budget"
)



m2.optimize()
print("\n--- Optimal shipment plan ---")
for p in pairs:
    print(f"{p[0]:8s} -> {p[1]:8s}: {x[p].X:.2f} (thousand units)")

print(f"\nOptimal total net profit: {m2.objVal:.2f} kkr (thousand Kr)")



Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 15 rows, 14 columns and 28 nonzeros (Max)
Model fingerprint: 0xbc3cefbf
Model has 14 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [6e-01, 2e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 4e+04]

Presolve removed 15 rows and 14 columns
Presolve time: 0.02s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.6489672e+04   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Optimal objective  3.648967193e+04

--- Optimal shipment plan ---
Panda    -> Poland  : 75.00 (thousand units)
Panda    -> Italy   : 35.00 (thousand units)
Panda    -> US      : 40.00 (thousand units)
Panda    -> Sweden  : 2.00 (thou

In [44]:
import gurobipy as gp
from gurobipy import Model, GRB
import numpy as np

tax = np.array([0.3, 0.15, 0.2, 0.3])
material_cost = np.array([0.57, 0.6, 0.55, 0.45])
salary = np.array([20000, 11000, 20000, 26000])
manhour = np.array([40, 45, 38, 100])
min_prod = np.array([120000, 100000, 80000, 15000])

price = np.array([[86e3, 92e3, 100e3, 0],
                  [106e3, 136e3, 150e3, 427e3],
                  [150e3, 170e3, 0, 550e3],
                  [112e3, 150e3, 170e3, 500e3]])
price_home = np.array([106e3, 92e3, 150e3, 427e3])
delivery_cost = np.array([[800, 0, 12e3, 0],
                          [0, 1000, 0, 0],
                          [3500, 2800, 0, 5000],
                          [2000, 1600, 2200, 2500]])
demand = np.array([[75e3, 20e3, 10e3, 0],
                   [35e3, 40e3, 80e3, 8e3],
                   [40e3, 50e3, 0, 3e3],
                   [2e3, 5e3, 1e3, 1e3]])
export_tax = np.array([[0, 0, 2.5e-2, 0]]).T

man_cost = salary * manhour / 160
material_cost = material_cost * price_home
total_cost = man_cost + material_cost + delivery_cost
profit = price - total_cost - price*export_tax
net_income = (1-tax) * profit

model = Model("Task_2")

# variables
country_num = 4
car_num = 4
x = model.addVars(country_num, car_num, vtype=GRB.INTEGER, name="x")

# objective function
net_income_allcars = gp.quicksum(net_income[i, j] * x[i, j] for i in range(4) for j in range(4))
model.setObjective(net_income_allcars, GRB.MAXIMIZE)

# budget constraint
total_cost_allcars = gp.quicksum(total_cost[i, j] * x[i, j] for i in range(4) for j in range(4))
model.addConstr(total_cost_allcars <= 40e9, "budget_constraint")

# minimum production
for j in range(car_num):
    model.addConstr(gp.quicksum(x[i, j] for i in range(4)) >= min_prod[j], name=f"minimum_production_{j}")

# minimum demand
for i in range(4):
    for j in range(4):
        model.addConstr(x[i, j] >= demand[i, j], name=f"minimum_demand_{i}_{j}")

# no export of Musa & Giulia
model.addConstr(x[0, 3] == 0, name="Giulia2Poland")
model.addConstr(x[2, 2] == 0, name="Musa2US")


model.optimize()

if model.status == GRB.OPTIMAL:
    x_sol = np.array([[x[i, j].X for j in range(4)] for i in range(4)])
    print("Optimal solution (x matrix):")
    print(x_sol)

    print(f"Optimal objective value: {model.objVal}")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 23 rows, 16 columns and 50 nonzeros (Max)
Model fingerprint: 0xe20de6fb
Model has 16 linear objective coefficients
Variable types: 0 continuous, 16 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+05]
  Objective range  [6e+02, 2e+05]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+03, 4e+10]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.

Found heuristic solution: objective 3.025433e+10
Presolve removed 21 rows and 9 columns
Presolve time: 0.01s
Presolved: 2 rows, 7 columns, 10 nonzeros
Found heuristic solution: objective 2.574420e+10
Variable types: 0 continuous, 7 integer (0 binary)

Root relaxation: objective 3.623547e+10, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Cur

Task 3 Multi-line production

Since the foam is composed of three components, and the rate is 1:1:1. Therefore, the maximum quantities of foam produced is decided by the minumum component. If the decision variable is x_ij, i represents component 1,2,3 and j represents which line produces. Then, we introduce a variable F, the total produced quantities of foam, the linear objective function is MAX F, with three lower than constraints.

In [22]:
import gurobipy as gp
from gurobipy import GRB
line=[1,2,3,4]
component=[1,2,3]
capacity={1:100,2:150,3:80,4:200}
rate={
    (1,1):10,(1,2):15,(1,3):5,
    (2,1):15,(2,2):10,(2,3):5,
    (3,1):20,(3,2):5,(3,3):10,
    (4,1):10,(4,2):15,(4,3):20,
}
#(lines,component):rate
m3=gp.Model("Multi-line production")
x=m3.addVars(line,component,name="hours")
F=m3.addVar(name="foam")
m3.setObjective(F,GRB.MAXIMIZE)
for j in component:
    m3.addConstr(F<=gp.quicksum(rate[i,j]*x[i,j] for i in line),f"foam_limit_comp{j}")
for i in line:
    m3.addConstr(gp.quicksum(x[i,j] for j in component)<=capacity[i],f"capacity_line{i}")
m3.optimize()
print("\n--- Hours allocated ---")
for l in line:
    for j in component:
        print(f"Line {l}, Component {j}: {x[l,j].X:.2f} hours")

print(f"\nMax foam produced: {F.X:.2f} units")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 13 columns and 27 nonzeros (Max)
Model fingerprint: 0xefff5a9a
Model has 1 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+01, 2e+02]

Presolve time: 0.01s
Presolved: 7 rows, 13 columns, 27 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.6000000e+31   3.000000e+30   1.600000e+01      0s
       9    2.9241379e+03   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.02 seconds (0.00 work units)
Optimal objective  2.924137931e+03

--- Hours allocated ---
Line 1, Component 1: 0.00 hours
Line 1, Component 2: 100.00 hours
Line 1, Component 3: 0.00 hours
Line 2, Component 1: 88.28 hours
Line 2, Compone